In [6]:
import os

In [7]:
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"

In [9]:
import wandb
import numpy as np
import random
import tensorflow as tf

from tensorflow.keras.datasets import cifar100
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from tensorflow import keras as k

In [10]:
# setting random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

wandb.login()

True

In [11]:
from wandb.integration.keras import WandbMetricsLogger, WandbModelCheckpoint

# --- custom callbacks -------

# logging the learning rate to wandb each epoch
class LogLRCallback(k.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        opt = self.model.optimizer
        lr = opt.learning_rate
        lr_val = float(lr.numpy() if hasattr(lr, "numpy") else lr)
        wandb.log({"lr": lr_val})

# logging a table of sample predictions with images and each epoch
class LogSamplesCallback(k.callbacks.Callback):
    def __init__(self, x, y, labels, max_rows=32):
        super().__init__()
        self.x = x[:max_rows]
        self.y = y[:max_rows]
        self.labels = labels

    def on_epoch_end(self, epoch, logs=None):
        preds = self.model.predict(self.x, verbose=0)
        y_true = np.argmax(self.y, axis=1)
        y_pred = np.argmax(preds, axis=1)

        table = wandb.Table(columns=["image", "y_true", "y_pred", "correct", "p(y_pred)"])
        for i in range(len(self.x)):
            # converting back to uint8 for displaying rgb images
            img = (self.x[i] * 255).astype(np.uint8)
            table.add_data(
                wandb.Image(img),
                self.labels[y_true[i]],
                self.labels[y_pred[i]],
                bool(y_true[i] == y_pred[i]),
                float(np.max(preds[i])),
            )
        wandb.log({f"samples/epoch_{epoch+1}": table})

# logging a confusion matrix from the val set for each epoch
class ConfusionMatrixCallback(k.callbacks.Callback):
    def __init__(self, x_val, y_val, labels):
        super().__init__()
        self.x_val = x_val
        self.y_val = y_val
        self.labels = labels

    def on_epoch_end(self, epoch, logs=None):
        preds = self.model.predict(self.x_val, verbose=0)
        y_true = np.argmax(self.y_val, axis=1)
        y_pred = np.argmax(preds, axis=1)
        cm_plot = wandb.plot.confusion_matrix(
            probs=None,
            y_true=y_true,
            preds=y_pred,
            class_names=self.labels,
        )
        wandb.log({"confusion_matrix": cm_plot})

# logging per-class accuracy as a bar chart
class PerClassAccuracyCallback(k.callbacks.Callback):
    def __init__(self, x_val, y_val, labels):
        super().__init__()
        self.x_val = x_val
        self.y_val = y_val
        self.labels = labels

    def on_epoch_end(self, epoch, logs=None):
        preds = self.model.predict(self.x_val, verbose=0)
        y_true = np.argmax(self.y_val, axis=1)
        y_pred = np.argmax(preds, axis=1)
        table = wandb.Table(columns=["class", "accuracy"])
        for i, label in enumerate(self.labels):
            mask = (y_true == i)
            acc = float((y_pred[mask] == i).mean()) if mask.sum() > 0 else 0.0
            table.add_data(label, acc)
        wandb.log({"per_class_accuracy": wandb.plot.bar(
            table, "class", "accuracy",
            title=f"Per-Class Accuracy (Epoch {epoch+1})")})

# trainer
class CIFAR100Trainer:
    def __init__(self, project_name="Lab5", run_name="cifar100_deep_cnn"):
        self.cfg = dict(
            dropout=0.35,
            conv1_filters=32,
            # second and third conv blocks
            conv2_filters=64,
            conv3_filters=128,
            # dense hidden layer
            dense_units=256,
            learn_rate=0.001,
            epochs=15,
            batch_size=128,
            sample_train=15000,
            sample_test=5000,
            val_fraction=0.15,
            augmentation=True,
        )

        self.run = wandb.init(
            project=project_name,
            name=run_name,
            config=self.cfg,
            settings=wandb.Settings(start_method="thread"),
        )
        self.config = wandb.config

        self.labels = [
            "aquatic_mammals", "fish", "flowers", "food_containers",
            "fruit_vegetables", "household_electrical", "household_furniture",
            "insects", "large_carnivores", "large_outdoor_man-made",
            "large_outdoor_natural", "large_omnivores_herbivores",
            "medium_mammals", "non-insect_invertebrates", "people",
            "reptiles", "small_mammals", "trees", "vehicles_1", "vehicles_2",
        ]
        self._prepare_data()

    def _prepare_data(self):
        (xtr_full, ytr_full), (xte_full, yte_full) = cifar100.load_data(label_mode="coarse")

        n_tr = self.config.sample_train
        n_te = self.config.sample_test
        # normalizing pixel values to [0, 1]
        xtr = xtr_full[:n_tr].astype("float32") / 255.0
        ytr = ytr_full[:n_tr].flatten()

        # train and validation split
        X_train, X_val, y_train_raw, y_val_raw = train_test_split(
            xtr, ytr, test_size=self.config.val_fraction,
            stratify=ytr, random_state=SEED
        )

        # test set
        X_test = xte_full[:n_te].astype("float32") / 255.0
        y_test_raw = yte_full[:n_te].flatten()

        # storing data
        self.X_train = X_train
        self.X_val   = X_val
        self.X_test  = X_test
        # one-hot encode labels
        self.y_train = to_categorical(y_train_raw, 20)
        self.y_val   = to_categorical(y_val_raw, 20)
        self.y_test  = to_categorical(y_test_raw, 20)
        # keeping labels for classification report
        self.y_test_raw = y_test_raw
        self.num_classes = 20

        print(f"Train: {self.X_train.shape} | Val: {self.X_val.shape} | Test: {self.X_test.shape}")

        self.datagen = None
        if self.config.augmentation:
            self.datagen = ImageDataGenerator(
                rotation_range=15,
                width_shift_range=0.1,
                height_shift_range=0.1,
                horizontal_flip=True,
            )
            self.datagen.fit(self.X_train)

    # added 3 conv blocks with batchnorm
    def _build_model(self):
        
        inputs = k.Input(shape=(32, 32, 3))  # rgb

        # block 1 — 32 filters
        x = k.layers.Conv2D(self.config.conv1_filters, (3, 3), padding="same")(inputs)
        x = k.layers.BatchNormalization()(x)
        x = k.layers.Activation("relu")(x)
        x = k.layers.Conv2D(self.config.conv1_filters, (3, 3), padding="same")(x)
        x = k.layers.BatchNormalization()(x)
        x = k.layers.Activation("relu")(x)
        x = k.layers.MaxPooling2D((2, 2))(x)
        x = k.layers.Dropout(self.config.dropout * 0.5)(x)
        # block 2 — 64 filters
        x = k.layers.Conv2D(self.config.conv2_filters, (3, 3), padding="same")(x)
        x = k.layers.BatchNormalization()(x)
        x = k.layers.Activation("relu")(x)
        x = k.layers.Conv2D(self.config.conv2_filters, (3, 3), padding="same")(x)
        x = k.layers.BatchNormalization()(x)
        x = k.layers.Activation("relu")(x)
        x = k.layers.MaxPooling2D((2, 2))(x)
        x = k.layers.Dropout(self.config.dropout)(x)
        # block 3 — 128 filters
        x = k.layers.Conv2D(self.config.conv3_filters, (3, 3), padding="same")(x)
        x = k.layers.BatchNormalization()(x)
        x = k.layers.Activation("relu")(x)
        x = k.layers.Conv2D(self.config.conv3_filters, (3, 3), padding="same")(x)
        x = k.layers.BatchNormalization()(x)
        x = k.layers.Activation("relu")(x)
        x = k.layers.MaxPooling2D((2, 2))(x)
        x = k.layers.Dropout(self.config.dropout)(x)

        # pooling
        x = k.layers.GlobalAveragePooling2D()(x)
        x = k.layers.Dense(self.config.dense_units, activation="relu")(x)
        x = k.layers.Dropout(self.config.dropout)(x)
        outputs = k.layers.Dense(self.num_classes, activation="softmax")(x)
        model = k.Model(inputs, outputs)
        model.compile(
            loss="categorical_crossentropy",
            optimizer=k.optimizers.Adam(learning_rate=self.config.learn_rate),
            metrics=["accuracy"],
        )
        return model

    def _log_model_artifact(self, model):
        # saving model summary and trained model as a wandb artifact
        summary_lines = []
        model.summary(print_fn=summary_lines.append)
        summary_txt = "\n".join(summary_lines)
        os.makedirs("artifacts", exist_ok=True)
        with open("artifacts/model_summary.txt", "w") as f:
            f.write(summary_txt)

        model_path = "artifacts/cifar100_model.h5"
        model.save(model_path)

        art = wandb.Artifact("cifar100_deep_cnn", type="model")
        art.add_file("artifacts/model_summary.txt")
        art.add_file(model_path)
        self.run.log_artifact(art)

    def train(self):
        model = self._build_model()
        callbacks = [
            # wandb logging
            WandbMetricsLogger(log_freq=10),
            WandbModelCheckpoint("checkpoints/model-{epoch:02d}.h5", save_weights_only=False),
            # custom callbacks
            LogLRCallback(),
            LogSamplesCallback(self.X_val, self.y_val, self.labels, max_rows=32),
            ConfusionMatrixCallback(self.X_val, self.y_val, self.labels),
            # per-class accuracy bar chart
            PerClassAccuracyCallback(self.X_val, self.y_val, self.labels),
            #early stopping
            k.callbacks.EarlyStopping(
                monitor="val_loss", patience=5,
                restore_best_weights=True, verbose=1),

            k.callbacks.ReduceLROnPlateau(
                monitor="val_loss", factor=0.5,
                patience=3, min_lr=1e-6, verbose=1),
        ]

        # training
        if self.datagen:
            model.fit(
                self.datagen.flow(self.X_train, self.y_train,
                                  batch_size=self.config.batch_size),
                validation_data=(self.X_val, self.y_val),
                epochs=self.config.epochs,
                steps_per_epoch=len(self.X_train) // self.config.batch_size,
                callbacks=callbacks,
                verbose=1,
            )
        else:
            model.fit(
                self.X_train, self.y_train,
                validation_data=(self.X_val, self.y_val),
                epochs=self.config.epochs,
                batch_size=self.config.batch_size,
                callbacks=callbacks,
                verbose=1,
            )

        # evaluating on test set
        loss, acc = model.evaluate(self.X_test, self.y_test, verbose=0)

        preds = np.argmax(model.predict(self.X_test, verbose=0), axis=1)
        precision = precision_score(self.y_test_raw, preds, average="weighted", zero_division=0)
        recall = recall_score(self.y_test_raw, preds, average="weighted", zero_division=0)
        f1 = f1_score(self.y_test_raw, preds, average="weighted", zero_division=0)

        print(f"\nTest Loss      = {loss:.4f}")
        print(f"Test Accuracy  = {acc:.4f}")
        print(f"Test Precision = {precision:.4f}")
        print(f"Test Recall    = {recall:.4f}")
        print(f"Test F1        = {f1:.4f}")

        # logging final metrics to wandb
        wandb.log({"final/loss": loss, "final/accuracy": acc,
                   "final/precision": precision, "final/recall": recall, "final/f1": f1})

        # classification report
        print("\n" + classification_report(self.y_test_raw, preds,
                                           target_names=self.labels, zero_division=0))

        # saving model artifact to wandb
        self._log_model_artifact(model)

        self.run.finish()


# running the trainer
CIFAR100Trainer().train()


/opt/anaconda3/envs/mlops_project/lib/python3.12/site-packages/keras/src/datasets/cifar.py:18: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  d = cPickle.load(f, encoding="bytes")


Train: (12750, 32, 32, 3) | Val: (2250, 32, 32, 3) | Test: (5000, 32, 32, 3)
Epoch 1/15
99/99 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step - accuracy: 0.1195 - loss: 2.8597

99/99 ━━━━━━━━━━━━━━━━━━━━ 18s 156ms/step - accuracy: 0.1622 - loss: 2.6928 - val_accuracy: 0.0520 - val_loss: 3.2983 - learning_rate: 0.0010
Epoch 2/15
 1/99 ━━━━━━━━━━━━━━━━━━━━ 12s 129ms/step - accuracy: 0.2344 - loss: 2.5120

/opt/anaconda3/envs/mlops_project/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


99/99 ━━━━━━━━━━━━━━━━━━━━ 3s 30ms/step - accuracy: 0.2344 - loss: 2.5120 - val_accuracy: 0.0520 - val_loss: 3.2922 - learning_rate: 0.0010
Epoch 3/15
71/99 ━━━━━━━━━━━━━━━━━━━━ 3s 113ms/step - accuracy: 0.2296 - loss: 2.4769

...

99/99 ━━━━━━━━━━━━━━━━━━━━ 14s 146ms/step - accuracy: 0.3563 - loss: 2.0408 - val_accuracy: 0.3738 - val_loss: 1.9951 - learning_rate: 5.0000e-04
Epoch 14/15
 1/99 ━━━━━━━━━━━━━━━━━━━━ 12s 125ms/step - accuracy: 0.3750 - loss: 1.8953

99/99 ━━━━━━━━━━━━━━━━━━━━ 3s 33ms/step - accuracy: 0.3750 - loss: 1.8953 - val_accuracy: 0.3653 - val_loss: 2.0305 - learning_rate: 5.0000e-04
Epoch 15/15
99/99 ━━━━━━━━━━━━━━━━━━━━ 0s 114ms/step - accuracy: 0.3661 - loss: 2.0098

99/99 ━━━━━━━━━━━━━━━━━━━━ 14s 146ms/step - accuracy: 0.3689 - loss: 1.9934 - val_accuracy: 0.3711 - val_loss: 1.9992 - learning_rate: 5.0000e-04
Restoring model weights from the end of the best epoch: 13.

Test Loss      = 1.9883
Test Accuracy  = 0.3732
Test Precision = 0.4148
Test Recall    = 0.3732
Test F1        = 0.3654

                            precision    recall  f1-score   support

           aquatic_mammals       0.43      0.22      0.29       243
                      fish       0.38      0.36      0.37       237
                   flowers       0.56      0.60      0.58       234
           food_containers       0.34      0.14      0.19       249
          fruit_vegetables       0.70      0.39      0.50       257
      household_electrical       0.48      0.15      0.23       235
       household_furniture       0.71      0.39      0.50       266
                   insects       0.21      0.61      0.32       250
          large_carnivores       0.27      0.44      0.34  

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


batch/accuracy,▁▁▂▂▂▄▄▄▄▇▅▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇███▇█▇▇▇█
batch/batch_step,▁▁▁▁▂▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▄▄▄▅▅▅▅▆▆▆▇▇▇▇▇▇▇██
batch/learning_rate,███████████████▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
batch/loss,█▆▆▆▄▄▄▄▄▄▃▃▃▃▃▂▂▃▂▂▂▂▂▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁
epoch/accuracy,▁▃▃▆▄▃▆▆▆▆▇▇▇██
epoch/epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
epoch/learning_rate,█████▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▆▆▅▅▆▄▃▃▂▃▂▂▁▂
epoch/val_accuracy,▁▁▂▂▁▁▃▃▄▄▆▆███
epoch/val_loss,▅▅▅▅██▄▄▃▃▂▂▁▁▁
+6,...
